# AI6103 Hyperparameter Study

深度学习作业：研究超参数对 EfficientNet-B0 的影响

**实验内容：**
- Section 2: 数据预处理
- Section 3: 学习率实验 (15 epochs × 3)
- Section 4: 学习率调度 (300 epochs × 2)
- Section 5: 权重衰减 (300 epochs × 2)
- Section 6: Mixup 数据增强 (300 epochs × 2)

## 1. 环境设置

In [ ]:
# 检查 GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 挂载 Google Drive (用于保存结果)
from google.colab import drive
drive.mount('/content/drive')

# 创建工作目录
!mkdir -p /content/drive/MyDrive/AI6103_Experiments

In [ ]:
# 克隆代码仓库
%cd /content
!rm -rf ai6103-hyperparameter-study
!git clone https://github.com/Pissohappy/ai6103-hyperparameter-study.git
%cd ai6103-hyperparameter-study
!ls -la

In [ ]:
# 安装依赖
!pip install -q kagglehub tqdm seaborn

## 2. 下载并准备数据集

In [ ]:
# 方法 1: 使用 kagglehub 直接下载
import kagglehub
import os
import shutil

# 下载数据集
print("Downloading Food-11 dataset...")
dataset_path = kagglehub.dataset_download('vermaavi/food11')
print(f"Dataset downloaded to: {dataset_path}")

# 设置数据目录
data_dir = '/content/data'
os.makedirs(data_dir, exist_ok=True)

# 移动数据到正确位置
if os.path.exists(os.path.join(dataset_path, 'training')):
    # 数据已经解压好
    for folder in ['training', 'validation', 'evaluation']:
        src = os.path.join(dataset_path, folder)
        dst = os.path.join(data_dir, folder)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print(f"Copied {folder}")
else:
    # 可能是 zip 文件
    import zipfile
    for f in os.listdir(dataset_path):
        if f.endswith('.zip'):
            with zipfile.ZipFile(os.path.join(dataset_path, f), 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            print(f"Extracted {f}")

# 验证数据
print("\nDataset structure:")
for folder in ['training', 'validation', 'evaluation']:
    path = os.path.join(data_dir, folder)
    if os.path.exists(path):
        num_images = sum([len(files) for _, _, files in os.walk(path)])
        print(f"  {folder}: {num_images} images")

## 3. 更新配置文件

In [ ]:
# 更新 config.py 中的路径
config_content = '''"""
Configuration for AI6103 experiments
"""
import os

# Paths
BASE_DIR = '/content/ai6103-hyperparameter-study'
DATA_DIR = '/content/data'
OUTPUT_DIR = '/content/drive/MyDrive/AI6103_Experiments/outputs'

# Dataset
IMAGE_SIZE = 100
NUM_CLASSES = 11

# Training defaults
DEFAULT_BATCH_SIZE = 128
DEFAULT_MOMENTUM = 0.9
DEFAULT_EPOCHS = 15

# Device
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
'''

with open('config.py', 'w') as f:
    f.write(config_content)

print("Config updated!")
print(f"Data directory: /content/data")
print(f"Output directory: /content/drive/MyDrive/AI6103_Experiments/outputs")

## 4. 导入模块

In [ ]:
import sys
sys.path.insert(0, '/content/ai6103-hyperparameter-study')

import os
import json
import torch
from config import DATA_DIR, OUTPUT_DIR, DEVICE
from data import compute_dataset_stats, get_dataloaders
from model import get_efficientnet_b0
from train import Trainer
from utils import (plot_training_curves, plot_lr_comparison, plot_beta_distribution,
                   get_final_results, print_results_table, save_results)

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"All modules loaded!")
print(f"Device: {DEVICE}")

---
## Section 2: 数据预处理

计算训练集的 RGB 均值和标准差

In [ ]:
# 计算数据集统计信息
train_path = os.path.join(DATA_DIR, 'training')
mean, std = compute_dataset_stats(train_path)

# 保存统计信息
stats = {'mean': mean, 'std': std}
save_results(stats, os.path.join(OUTPUT_DIR, 'dataset_stats.json'))

print("\n" + "="*60)
print("📝 在报告中报告这些数值：")
print(f"   Mean (R, G, B): {mean}")
print(f"   Std (R, G, B): {std}")
print("="*60)

---
## Section 3: 学习率实验

测试 LR = 0.1, 0.025, 0.001，各 15 epochs

In [ ]:
# Section 3: 学习率实验
learning_rates = [0.1, 0.025, 0.001]
epochs = 15

train_loader, val_loader, test_loader = get_dataloaders(
    mean=mean, std=std, augment=True
)

section3_results = {}
section3_histories = []
section3_labels = []

for lr in learning_rates:
    print(f"\n{'='*60}")
    print(f"Training with LR = {lr}")
    print(f"{'='*60}")
    
    model = get_efficientnet_b0(pretrained=False)
    trainer = Trainer(model, train_loader, val_loader, DEVICE)
    
    experiment_name = f"section3_lr_{lr}"
    history = trainer.train(
        epochs=epochs,
        lr=lr,
        momentum=0.9,
        weight_decay=0.0,
        scheduler_type=None,
        save_dir=OUTPUT_DIR,
        experiment_name=experiment_name
    )
    
    section3_histories.append(history)
    section3_labels.append(f"LR={lr}")
    section3_results[experiment_name] = get_final_results(history)
    
    # 清理显存
    del model, trainer
    torch.cuda.empty_cache()

# 绘制对比图
plot_lr_comparison(
    section3_histories, section3_labels,
    title="Learning Rate Comparison (Section 3)",
    save_path=os.path.join(OUTPUT_DIR, "section3_comparison.png"),
    show=False
)

# 打印结果
print_results_table(section3_results)
save_results(section3_results, os.path.join(OUTPUT_DIR, "section3_results.json"))

# 找最佳学习率
best_lr_name = max(section3_results.items(), key=lambda x: x[1]['final_val_acc'])[0]
print(f"\n🏆 Best Learning Rate: {best_lr_name}")

---
## Section 4: 学习率调度

比较固定学习率 vs Cosine Annealing，300 epochs

In [ ]:
# Section 4: 学习率调度
# 使用 Section 3 找到的最佳学习率
best_lr = float(best_lr_name.split('_')[-1])
epochs = 300

print(f"Using best LR = {best_lr}")

section4_results = {}
section4_histories = []
section4_labels = []

# 实验 1: 固定学习率
print(f"\n{'='*60}")
print("Experiment: Fixed Learning Rate")
print(f"{'='*60}")

model = get_efficientnet_b0(pretrained=False)
trainer = Trainer(model, train_loader, val_loader, DEVICE)

history_fixed = trainer.train(
    epochs=epochs,
    lr=best_lr,
    momentum=0.9,
    weight_decay=0.0,
    scheduler_type=None,
    save_dir=OUTPUT_DIR,
    experiment_name="section4_fixed_lr"
)

section4_histories.append(history_fixed)
section4_labels.append("Fixed LR")
section4_results["Fixed LR"] = get_final_results(history_fixed)

del model, trainer
torch.cuda.empty_cache()

# 实验 2: Cosine Annealing
print(f"\n{'='*60}")
print("Experiment: Cosine Annealing")
print(f"{'='*60}")

model = get_efficientnet_b0(pretrained=False)
trainer = Trainer(model, train_loader, val_loader, DEVICE)

history_cosine = trainer.train(
    epochs=epochs,
    lr=best_lr,
    momentum=0.9,
    weight_decay=0.0,
    scheduler_type='cosine',
    save_dir=OUTPUT_DIR,
    experiment_name="section4_cosine"
)

section4_histories.append(history_cosine)
section4_labels.append("Cosine Annealing")
section4_results["Cosine Annealing"] = get_final_results(history_cosine)

del model, trainer
torch.cuda.empty_cache()

# 绘制对比图
plot_lr_comparison(
    section4_histories, section4_labels,
    title="Learning Rate Schedule Comparison (Section 4)",
    save_path=os.path.join(OUTPUT_DIR, "section4_comparison.png"),
    show=False
)

print_results_table(section4_results)
save_results(section4_results, os.path.join(OUTPUT_DIR, "section4_results.json"))

---
## Section 5: 权重衰减

测试 λ = 5e-4 和 1e-4，使用 Cosine Annealing，300 epochs

In [ ]:
# Section 5: 权重衰减
weight_decays = [5e-4, 1e-4]
epochs = 300

section5_results = {}
section5_histories = []
section5_labels = []

for wd in weight_decays:
    print(f"\n{'='*60}")
    print(f"Training with Weight Decay = {wd}")
    print(f"{'='*60}")
    
    model = get_efficientnet_b0(pretrained=False)
    trainer = Trainer(model, train_loader, val_loader, DEVICE)
    
    experiment_name = f"section5_wd_{wd}"
    history = trainer.train(
        epochs=epochs,
        lr=best_lr,
        momentum=0.9,
        weight_decay=wd,
        scheduler_type='cosine',
        save_dir=OUTPUT_DIR,
        experiment_name=experiment_name
    )
    
    section5_histories.append(history)
    section5_labels.append(f"WD={wd}")
    section5_results[f"WD={wd}"] = get_final_results(history)
    
    del model, trainer
    torch.cuda.empty_cache()

# 绘制对比图
plot_lr_comparison(
    section5_histories, section5_labels,
    title="Weight Decay Comparison (Section 5)",
    save_path=os.path.join(OUTPUT_DIR, "section5_comparison.png"),
    show=False
)

print_results_table(section5_results)
save_results(section5_results, os.path.join(OUTPUT_DIR, "section5_results.json"))

---
## Section 6: Mixup 数据增强

比较使用和不使用 Mixup，300 epochs

In [ ]:
# 绘制 Beta 分布
plot_beta_distribution(
    alpha=0.2,
    save_path=os.path.join(OUTPUT_DIR, "section6_beta_distribution.png"),
    show=False
)
print("Beta distribution plot saved!")

In [ ]:
# Section 6: Mixup 实验
epochs = 300
mixup_alpha = 0.2

section6_results = {}
section6_histories = []
section6_labels = []

# 使用最佳权重衰减
best_wd = 5e-4

# 实验 1: 不使用 Mixup
print(f"\n{'='*60}")
print("Experiment: Without Mixup")
print(f"{'='*60}")

model = get_efficientnet_b0(pretrained=False)
trainer = Trainer(model, train_loader, val_loader, DEVICE)

history_no_mixup = trainer.train(
    epochs=epochs,
    lr=best_lr,
    momentum=0.9,
    weight_decay=best_wd,
    scheduler_type='cosine',
    use_mixup=False,
    save_dir=OUTPUT_DIR,
    experiment_name="section6_no_mixup"
)

section6_histories.append(history_no_mixup)
section6_labels.append("No Mixup")
section6_results["No Mixup"] = get_final_results(history_no_mixup)

del model, trainer
torch.cuda.empty_cache()

# 实验 2: 使用 Mixup
print(f"\n{'='*60}")
print("Experiment: With Mixup")
print(f"{'='*60}")

model = get_efficientnet_b0(pretrained=False)
trainer = Trainer(model, train_loader, val_loader, DEVICE)

history_mixup = trainer.train(
    epochs=epochs,
    lr=best_lr,
    momentum=0.9,
    weight_decay=best_wd,
    scheduler_type='cosine',
    use_mixup=True,
    mixup_alpha=mixup_alpha,
    save_dir=OUTPUT_DIR,
    experiment_name="section6_mixup"
)

section6_histories.append(history_mixup)
section6_labels.append("Mixup")
section6_results["Mixup"] = get_final_results(history_mixup)

del model, trainer
torch.cuda.empty_cache()

# 绘制对比图
plot_lr_comparison(
    section6_histories, section6_labels,
    title="Mixup Comparison (Section 6)",
    save_path=os.path.join(OUTPUT_DIR, "section6_comparison.png"),
    show=False
)

print_results_table(section6_results)
save_results(section6_results, os.path.join(OUTPUT_DIR, "section6_results.json"))

---
## 最终测试集评估

In [ ]:
# 使用最佳模型评估测试集
print("\n" + "="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)

# 重新训练最佳配置 (或加载已保存的最佳模型)
model = get_efficientnet_b0(pretrained=False)
model = model.to(DEVICE)

# 使用最佳配置训练
trainer = Trainer(model, train_loader, val_loader, DEVICE)

# 用完整训练数据训练 (train + val) - 可选
# 或者直接用之前的最佳模型

criterion = torch.nn.CrossEntropyLoss()
from train import validate

test_loss, test_acc = validate(model, test_loader, criterion, DEVICE)
print(f"\n📊 Test Accuracy: {test_acc:.2f}%")
print(f"📊 Test Loss: {test_loss:.4f}")

# 保存最终结果
final_results = {
    'test_accuracy': test_acc,
    'test_loss': test_loss,
    'best_lr': best_lr,
    'best_weight_decay': best_wd,
    'mixup_alpha': mixup_alpha
}
save_results(final_results, os.path.join(OUTPUT_DIR, "final_test_results.json"))

---
## 查看所有结果

In [ ]:
# 列出所有生成的文件
print("\n📁 Generated files in Google Drive:")
!ls -la /content/drive/MyDrive/AI6103_Experiments/outputs/

print("\n📊 Summary of all experiments:")
print("\nSection 3 (Learning Rate):")
print_results_table(section3_results)

print("\nSection 4 (LR Schedule):")
print_results_table(section4_results)

print("\nSection 5 (Weight Decay):")
print_results_table(section5_results)

print("\nSection 6 (Mixup):")
print_results_table(section6_results)

In [ ]:
# 打包下载所有结果 (可选)
!cd /content/drive/MyDrive/AI6103_Experiments && zip -r results.zip outputs/
print("\n✅ Results zipped: /content/drive/MyDrive/AI6103_Experiments/results.zip")